In [1]:
%env CUDA_DEVICES = 1

import os
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ["CUDA_DEVICES"]

env: CUDA_DEVICES=1


In [2]:
import numpy as np
import pandas as pd 
import os
import gc
from tqdm import tqdm

from sklearn.preprocessing import MultiLabelBinarizer
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

# Import Data 

In [3]:
# Load train and test metadata
pa_train = pd.read_csv("./metadata/PA_metadata_train.csv", dtype={"speciesId":int})

mediterranean_species = pa_train[pa_train.region == 'MEDITERRANEAN'].speciesId.value_counts()
mediterranean_species = list(mediterranean_species.index[0:200])

continental_species = pa_train[pa_train.region == 'CONTINENTAL'].speciesId.value_counts()
continental_species = list(continental_species.index[0:200])

atlantic_species = pa_train[pa_train.region == 'ATLANTIC'].speciesId.value_counts()
atlantic_species = list(atlantic_species.index[0:200])

alpine_species = pa_train[pa_train.region == 'ALPINE'].speciesId.value_counts()
alpine_species = list(alpine_species.index[0:200])

all_species = list(set(mediterranean_species) | set(continental_species) | set(atlantic_species) | set(alpine_species))

pa_train = pa_train[pa_train.speciesId.isin(all_species)].reset_index(drop=True)

test = pd.read_csv("./metadata/PA_metadata_test.csv")
test = test.dropna(subset=["speciesId"])


elevation_train = pd.read_csv("./metadata/PA-train-elevation.csv")
elevation_test = pd.read_csv("./metadata/PA-test-elevation.csv")

bioclim_train = pd.read_csv("./metadata/PA-train-bioclimatic.csv")
bioclim_test = pd.read_csv("./metadata/PA-test-bioclimatic.csv")

landcover_train = pd.read_csv("./metadata/PA-train-landcover.csv")
landcover_test = pd.read_csv("./metadata/PA-test-landcover.csv")


print(pa_train.shape)
for dataset in [elevation_train, bioclim_train, landcover_train]:
    pa_train = pd.merge(pa_train, dataset, how = "left", left_on = "surveyId", right_on = "surveyId")
    print(pa_train.shape)
    del dataset
    
print(test.shape)
for dataset in [elevation_test, bioclim_test, landcover_test]:
    test = pd.merge(test, dataset, how = "left", left_on = "surveyId", right_on = "surveyId")
    print(test.shape)
    del dataset
    

_ = gc.collect()

test = test.drop(columns=["PlotObservationID_eva", "observer", "access", "surveyID", "coverHerbLayer", "coverMossLayer", "source", "coverTreeLayer", "coverShrubLayer", "Cover.abundance.scale", "datasetName", "Access.regime", "Expert.System"])

dict_groupby = {k:["first"] for k in pa_train.columns.drop(["speciesId", "surveyId"])} 
dict_groupby["speciesId"] = [list]

pa_train["speciesId"] = pa_train["speciesId"].astype(int)
pa_train = pa_train.groupby("surveyId").agg(dict_groupby).reset_index()
pa_train.columns = [x[0] for x in pa_train.columns]
print(pa_train.shape)

pa_train["speciesId"] = pa_train["speciesId"].apply(lambda x : list(set(x)))# pa_train.head(2)
pa_train["speciesId"] = pa_train["speciesId"].apply(lambda x : list(set(x)))# pa_train.head(2)

dict_groupby = {k:["first"] for k in test.columns.drop(["speciesId", "surveyId"])} 
dict_groupby["speciesId"] = [list]

test["speciesId"] = test["speciesId"].astype(int)
test = test.groupby("surveyId").agg(dict_groupby).reset_index()
test.columns = [x[0] for x in test.columns]
print(test.shape)

test["speciesId"] = test["speciesId"].apply(lambda x : list(set(x)))# test.head(2)
test["speciesId"] = test["speciesId"].apply(lambda x : list(set(x)))# test.head(2)


pd.set_option('mode.use_inf_as_na', True)

def preprocess(df_to_preprocess:pd.DataFrame) -> pd.DataFrame:
    
    df = df_to_preprocess.copy()
    df = df.drop(columns=["geoUncertaintyInM", "areaInM2", "country"])

    df['Elevation'] = df['Elevation'].fillna(df['Elevation'].mode()[0])

    for column in df.filter(regex='Bio').columns:
        df[column] = df[column].fillna(df[column].mode()[0])
        
    for column in df.filter(regex='Soilgrid').columns:
        df[column] = df[column].fillna(df[column].mode()[0])

    # columns_to_convert =['Elevation', 'HumanFootprint-Pasture1993', 'HumanFootprint-Pasture2009']
    columns_to_convert = ['Elevation']

    df[columns_to_convert] = df[columns_to_convert].astype(int)

    for name,values in df.filter(regex='Soilgrid').items():
        df[name] = df[name].astype(int)
    
    df["surveyId"]=df["surveyId"].astype(int).astype(str)
        
    df.fillna(0, inplace = True)
        
    return df

pa_train = preprocess(pa_train)
test = preprocess(test)

(1198096, 9)
(1198096, 10)
(1198096, 29)
(1198096, 42)
(93397, 22)
(93397, 23)
(93397, 42)
(93397, 55)
(88087, 42)
(4717, 42)


# Location + Bioclim + Landcover

In [4]:
import pandas as pd
from sklearn.metrics import roc_auc_score, recall_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from xgboost import XGBClassifier
import numpy as np
import time

start_time = time.time()
# Set a random seed for reproducibility
seed = 1
np.random.seed(seed)

metrics = {"AUC": [], "Recall": [], "F1": [], "Seed": []}

# Set up training and testing data
X_train = pa_train.copy().set_index("surveyId")
X_test = test.copy().set_index("surveyId")

# Label encoding
mlb = MultiLabelBinarizer()
y_train = pd.DataFrame(mlb.fit_transform(X_train["speciesId"]),
                       columns=mlb.classes_,
                       index=X_train.index)

# Drop unused columns from training and testing sets
X_train.drop(["speciesId", "region"], axis=1, inplace=True)
X_test.drop("region", axis=1, inplace=True)
X_test = X_test[X_train.columns]  # Align test columns with train columns

# Hyperparameters for the XGBoost model
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'colsample_bytree': 0.8,
    'learning_rate': 0.1,
    'max_depth': 9,
    'n_estimators': 200,
    'reg_alpha': 0.2,
    'reg_lambda': 0.8,
    'tree_method': 'hist',
}

# Train the model
model = XGBClassifier(**xgb_params, seed=seed)
model.fit(X_train, y_train)

# Predict probabilities for the test set
proba_predictions = model.predict_proba(X_test)
classes = mlb.classes_

# Prepare the test labels
y_test = pd.DataFrame(mlb.transform(test["speciesId"]),
                      columns=mlb.classes_,
                      index=X_test.index)

# Filter out constant columns (all zeros or all ones) in y_test and proba_predictions
zero_cols_targets = np.all(y_test == 0, axis=0)
ones_cols_targets = np.all(y_test == 1, axis=0)
zero_cols = zero_cols_targets | ones_cols_targets

filtered_targets = y_test.loc[:, ~zero_cols]
filtered_probas = pd.DataFrame(proba_predictions, columns=classes).loc[:, ~zero_cols]

auc_score = roc_auc_score(filtered_targets, filtered_probas, average="macro")

top_k = 25
top25_probas = filtered_probas.apply(lambda row: pd.Series(row.nlargest(top_k).index), axis=1)

# Convert these indices to a binary format for recall and F1 calculation
top25_predictions = pd.DataFrame(0, index=filtered_probas.index, columns=filtered_probas.columns)
for i, top_species in top25_probas.iterrows():
    top25_predictions.loc[i, top_species] = 1

# Calculate recall and F1 using only the top 25 predictions
recall = recall_score(filtered_targets, top25_predictions, average="samples")
f1 = f1_score(filtered_targets, top25_predictions, average="samples")

# Store metrics
metrics["Seed"].append(seed)
metrics["AUC"].append(auc_score)
metrics["Recall"].append(recall)
metrics["F1"].append(f1)

# Convert metrics dictionary to DataFrame for easy review
metrics_df = pd.DataFrame(metrics)
print(metrics_df)
print(f"Elapsad time: {time.time() - start_time}")

        AUC    Recall        F1  Seed
0  0.903876  0.489405  0.287846     1
Elapsad time: 596.4125196933746
